In [1]:
import sys
import os

PROJECT_ROOT = "/home/praneeth/dissertation/context-aware-magail-grf"
GRF_MARL_ROOT = "/home/praneeth/dissertation/GRF_MARL"

sys.path.insert(0, os.path.join(PROJECT_ROOT, "src", "grf_baseline"))
sys.path.insert(0, os.path.join(GRF_MARL_ROOT, "light_malib", "model", "gr_football", "enhanced_LightActionMask_5"))
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src", "metrics"))
sys.path.insert(0, GRF_MARL_ROOT)

import torch
import numpy as np
import gfootball.env as football_env
from minimal_state import MinimalState
from enhanced_LightActionMask_5 import FeatureEncoder

print("All imports OK")
print("PyTorch:", torch.__version__)

/home/praneeth/miniconda3/envs/dissertation-grf/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports OK
PyTorch: 1.13.1+cu117


In [2]:
env = football_env.create_environment(
    env_name='5_vs_5', representation='raw',
    number_of_left_players_agent_controls=4, render=False
)
raw_obs = env.reset()

print("Number of controlled agents:", len(raw_obs))
print("Keys in observation:", list(raw_obs[0].keys())[:8], "...")
print("steps_left:", raw_obs[0]['steps_left'])
print("score:", raw_obs[0]['score'])

Number of controlled agents: 4
Keys in observation: ['right_team', 'left_team_yellow_card', 'ball_owned_team', 'right_team_tired_factor', 'steps_left', 'left_team_direction', 'ball_owned_player', 'score'] ...
steps_left: 3001
score: [0, 0]


In [3]:
def compute_context(obs):
    """
    Extracts (T_norm, delta_score) from a raw GRF observation.
    T_norm: normalised time REMAINING (matches encoder's own convention,
        see enhanced_LightActionMask_5.py: steps_left / 3001).
    delta_score: our team's score minus opponent's score.
    """
    t_norm = obs['steps_left'] / 3001.0
    delta_score = obs['score'][0] - obs['score'][1]
    return t_norm, delta_score


# Test across several steps, confirm T_norm decreases monotonically,
# delta_score stays at 0 unless a goal actually happens
env.reset()
obs = raw_obs

context_log = []
for step in range(20):
    actions = env.action_space.sample()
    obs, reward, done, info = env.step(actions)
    t_norm, delta_score = compute_context(obs[0])
    context_log.append((step, t_norm, delta_score))
    if done:
        obs = env.reset()

print(f"{'step':>5} {'T_norm':>10} {'delta_score':>12}")
for step, t_norm, delta_score in context_log:
    print(f"{step:>5} {t_norm:>10.4f} {delta_score:>12}")

 step     T_norm  delta_score
    0     0.9997            0
    1     0.9993            0
    2     0.9990            0
    3     0.9987            0
    4     0.9983            0
    5     0.9980            0
    6     0.9977            0
    7     0.9973            0
    8     0.9970            0
    9     0.9967            0
   10     0.9963            0
   11     0.9960            0
   12     0.9957            0
   13     0.9953            0
   14     0.9950            0
   15     0.9947            0
   16     0.9943            0
   17     0.9940            0
   18     0.9937            0
   19     0.9933            0


In [4]:
import torch.nn as nn

# Confirm the isolated shape math before touching the real 192-dim architecture.
# Real encoder output is 192-dim; adding (T_norm, delta_score) makes 194-dim.

test_layer = nn.Linear(194, 256)
dummy_input = torch.randn(1, 194)
out = test_layer(dummy_input)
print("Single sample -- input shape:", dummy_input.shape, "-> output shape:", out.shape)

# Also confirm it works batched (needed later for training, and for the
# 4-agent batching optimisation flagged during today's throughput discussion)
dummy_batch = torch.randn(4, 194)
out_batch = test_layer(dummy_batch)
print("Batch of 4 agents -- input shape:", dummy_batch.shape, "-> output shape:", out_batch.shape)

# Now build the ACTUAL concatenation you'll use in the real pipeline:
# real 192-dim encoder output + 2-dim context vector
real_encoder_output = torch.randn(1, 192)  # stand-in for encoder.encode_each() output
t_norm, delta_score = compute_context(obs[0])
context_vec = torch.tensor([[t_norm, delta_score]], dtype=torch.float32)

context_extended_input = torch.cat([real_encoder_output, context_vec], dim=-1)
print("\nConcatenated real-shape test:")
print("  encoder output:", real_encoder_output.shape)
print("  context vector:", context_vec.shape)
print("  concatenated:  ", context_extended_input.shape, "(expect [1, 194])")

Single sample -- input shape: torch.Size([1, 194]) -> output shape: torch.Size([1, 256])
Batch of 4 agents -- input shape: torch.Size([4, 194]) -> output shape: torch.Size([4, 256])

Concatenated real-shape test:
  encoder output: torch.Size([1, 192])
  context vector: torch.Size([1, 2])
  concatenated:   torch.Size([1, 194]) (expect [1, 194])


In [5]:
from minimal_state import MinimalState

# Load the real, verified baseline checkpoint (this becomes both the
# starting point for fine-tuning AND the frozen pi* reference)
actor_path = os.path.join(
    GRF_MARL_ROOT, "light_malib", "trained_models", "gr_football",
    "5_vs_5", "PassingMain_v2", "actor.pt"
)
actor = torch.load(actor_path, map_location="cpu")
actor.eval()

# Get a REAL 192-dim encoded observation (not dummy data this time)
from enhanced_LightActionMask_5 import FeatureEncoder
encoder = FeatureEncoder()

env.reset()
raw_obs = env.reset()
state = MinimalState(n_player=5)
state.set_obs(raw_obs[0])
real_feat = encoder.encode_each(state)
real_feat_tensor = torch.as_tensor(real_feat, dtype=torch.float32).unsqueeze(0)  # (1, 192)

avail = encoder.get_available_actions(raw_obs[0], 0.0, [])
mask_tensor = torch.as_tensor(avail, dtype=torch.float32).unsqueeze(0)

# Path A: call pi* on the ORIGINAL 192-dim input directly (ground truth)
with torch.no_grad():
    action_a, _, logprob_a, _ = actor(real_feat_tensor, None, None, mask_tensor, False, None)

# Path B: simulate the fine-tuning-era flow -- build a 194-dim context-extended
# tensor (as the trainable policy would receive), then STRIP the context dims
# back off before feeding pi*, exactly as the frozen-reference call must do
t_norm, delta_score = compute_context(raw_obs[0])
context_vec = torch.tensor([[t_norm, delta_score]], dtype=torch.float32)
context_extended = torch.cat([real_feat_tensor, context_vec], dim=-1)  # (1, 194)

stripped_back = context_extended[:, :192]  # remove context dims before calling pi*
with torch.no_grad():
    action_b, _, logprob_b, _ = actor(stripped_back, None, None, mask_tensor, False, None)

print("Path A (direct 192-dim call)      -- action:", action_a.item(), " logprob:", logprob_a.item())
print("Path B (194-dim built, then stripped) -- action:", action_b.item(), " logprob:", logprob_b.item())
print("\nIdentical?", torch.equal(real_feat_tensor, stripped_back), "|",
      action_a.item() == action_b.item(), "|",
      abs(logprob_a.item() - logprob_b.item()) < 1e-6)

Path A (direct 192-dim call)      -- action: 14  logprob: 0.0
Path B (194-dim built, then stripped) -- action: 14  logprob: 0.0

Identical? True | True | True


In [6]:
class Discriminator(nn.Module):
    """
    Locked spec: ~150-dim input (positions + pairwise relative positions +
    ball state + action one-hot + context), sigmoid output via BCEWithLogits
    (so this returns raw logits, not probabilities -- loss function applies
    sigmoid internally for numerical stability).
    """
    def __init__(self, input_dim=150):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        return self.net(x)


# Shape test only -- dummy data, untrained weights
discriminator = Discriminator(input_dim=150)

dummy_batch = torch.randn(8, 150)  # batch of 8 transitions
logits = discriminator(dummy_batch)
print("Discriminator output shape:", logits.shape, "(expect [8, 1])")

probs = torch.sigmoid(logits)
print("Sample probabilities:", probs.squeeze().tolist()[:4])

Discriminator output shape: torch.Size([8, 1]) (expect [8, 1])
Sample probabilities: [0.5438774824142456, 0.5119829773902893, 0.511789083480835, 0.52985018491745]


In [7]:
torch.manual_seed(42)  # reproducible for this diagnostic

# Same spatial/action features (148 dims -- everything except the 2 context dims)
same_spatial_features = torch.randn(1, 148)

# Two very different contexts
ctx_early_neutral = torch.tensor([[0.1, 0.0]])   # T_norm=0.1 (early), delta_score=0
ctx_late_losing    = torch.tensor([[0.9, -2.0]])  # T_norm=0.9 (late), delta_score=-2

input_a = torch.cat([same_spatial_features, ctx_early_neutral], dim=-1)
input_b = torch.cat([same_spatial_features, ctx_late_losing], dim=-1)

print("Input A shape:", input_a.shape, "| Input B shape:", input_b.shape)

with torch.no_grad():
    logit_a = discriminator(input_a)
    logit_b = discriminator(input_b)

prob_a = torch.sigmoid(logit_a).item()
prob_b = torch.sigmoid(logit_b).item()

print(f"\nSame spatial features, context A (early-neutral):  logit={logit_a.item():.4f}  prob={prob_a:.4f}")
print(f"Same spatial features, context B (late-losing):     logit={logit_b.item():.4f}  prob={prob_b:.4f}")
print(f"\nDifference in output: {abs(prob_a - prob_b):.6f}")
print("Context reaches the network and changes output:", abs(prob_a - prob_b) > 1e-6)

Input A shape: torch.Size([1, 150]) | Input B shape: torch.Size([1, 150])

Same spatial features, context A (early-neutral):  logit=0.1521  prob=0.5379
Same spatial features, context B (late-losing):     logit=0.1718  prob=0.5428

Difference in output: 0.004904
Context reaches the network and changes output: True
